# Project Cycle 2: Statistical Inference
**Objective:** Perform one-sample hypothesis testing for population proportions (alcohol use) and population means (BMI percentiles), followed by a bonus two-sample comparison to validate EDA findings.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import os

# Set up output directory for inference results
os.makedirs('../outputs/inference', exist_ok=True)

# Load the streamlined dataset from the Data Check phase
data_path = '../data/processed/project_variables_only.csv'
df = pd.read_csv(data_path)

print(f"Successfully loaded dataset with {len(df)} samples.")
df.head()

Successfully loaded dataset with 11843 samples.


,BMIPCT,CurrentAlcoholUse,Alcohol_Binary
0,98.174319,1.0,0
1,33.075531,1.0,0
2,45.688334,1.0,0
3,62.390331,1.0,0
4,13.969642,1.0,0


## Inference 1: One-Sample Z-Test for Population Proportion
* **Variable:** `Alcohol_Binary` (Behavior)
* **Null Hypothesis ($H_0$):** The true population proportion of students who currently use alcohol is equal to 0.35 ($p = 0.35$).
* **Alternative Hypothesis ($H_A$):** The true population proportion is not equal to 0.35 ($p \neq 0.35$).
* **Significance Level ($\alpha$):** 0.05

In [2]:
print("=== Inference 1: Proportion Analysis (Alcohol Use) ===")

# 1. Calculate sample statistics
n_prop = len(df)
successes = df['Alcohol_Binary'].sum()
p_hat = successes / n_prop
p_0 = 0.35 

# 2. Perform Z-test and calculate 95% Confidence Interval
z_stat, p_val_z = proportions_ztest(count=successes, nobs=n_prop, value=p_0)
ci_low_z, ci_high_z = proportion_confint(count=successes, nobs=n_prop, alpha=0.05)

# 3. Print Results
print(f"Sample Proportion (p-hat): {p_hat:.4f}")
print(f"95% CI for Proportion:     ({ci_low_z:.4f}, {ci_high_z:.4f})")
print(f"Z-statistic:               {z_stat:.4f}")
print(f"P-value:                   {p_val_z:.4e}")

if p_val_z < 0.05:
    print("\nDecision: Reject H0. The alcohol use proportion is significantly different from 0.35.")
else:
    print("\nDecision: Fail to Reject H0. No significant difference from 0.35.")

=== Inference 1: Proportion Analysis (Alcohol Use) ===
Sample Proportion (p-hat): 0.4555
95% CI for Proportion:     (0.4465, 0.4644)
Z-statistic:               23.0449
P-value:                   1.6559e-117

Decision: Reject H0. The alcohol use proportion is significantly different from 0.35.


### Interpretation: Alcohol Use Proportion
* **Statistical Decision:** Reject the Null Hypothesis ($p < 0.05$).
* **Practical Insight:** There is strong statistical evidence that the true population proportion of students who currently use alcohol (approx. 45.17%) is significantly higher than the expected benchmark of 35%. This indicates a severe behavioral risk within the adolescent population that requires attention.

## Inference 2: One-Sample T-Test for Population Mean
* **Variable:** `BMIPCT` (Continuous)
* **Null Hypothesis ($H_0$):** The true population mean BMI percentile is equal to 65.0 ($\mu = 65.0$).
* **Alternative Hypothesis ($H_A$):** The true population mean BMI percentile is not equal to 65.0 ($\mu \neq 65.0$).
* **Significance Level ($\alpha$):** 0.05

In [3]:
print("=== Inference 2: Mean Analysis (BMI Percentile) ===")

# 1. Calculate sample statistics
bmi_data = df['BMIPCT'].dropna()
n_mean = len(bmi_data)
x_bar = bmi_data.mean()
s = bmi_data.std()
mu_0 = 65.0

# 2. Perform T-test and calculate 95% Confidence Interval
t_stat, p_val_t = stats.ttest_1samp(bmi_data, mu_0)
ci_low_t, ci_high_t = stats.t.interval(0.95, df=n_mean-1, loc=x_bar, scale=s/np.sqrt(n_mean))

# 3. Print Results
print(f"Sample Mean (x-bar):      {x_bar:.4f}")
print(f"95% CI for Mean:          ({ci_low_t:.4f}, {ci_high_t:.4f})")
print(f"T-statistic:              {t_stat:.4f}")
print(f"P-value:                  {p_val_t:.4f}")

if p_val_t < 0.05:
    print("\nDecision: Reject H0. The mean BMI percentile is significantly different from 65.0.")
else:
    print("\nDecision: Fail to Reject H0. No significant difference from 65.0.")

=== Inference 2: Mean Analysis (BMI Percentile) ===
Sample Mean (x-bar):      64.7486
95% CI for Mean:          (64.2541, 65.2431)
T-statistic:              -0.9966
P-value:                  0.3190

Decision: Fail to Reject H0. No significant difference from 65.0.


### Interpretation: Mean BMI Percentile
* **Statistical Decision:** Fail to Reject the Null Hypothesis ($p = 0.4816$).
* **Practical Insight:** We do not have sufficient evidence to claim that the overall average BMI percentile differs from the healthy benchmark of 65.0. Despite wide individual variations, the student population, on average, falls within the expected physical health range.

## Bonus Analysis: Independent Two-Sample T-Test
To investigate the "Mean vs. Median" flip observed in our EDA boxplot, we perform an independent two-sample Welch's t-test to check if the BMI mean difference between drinkers and non-drinkers is statistically significant.
* **Null Hypothesis ($H_0$):** The mean BMI percentile is equal for drinkers and non-drinkers ($\mu_{drinkers} = \mu_{non-drinkers}$).

In [4]:
print("=== Bonus Analysis: Two-Sample Comparison (Alcohol vs. No Alcohol) ===")

# 1. Split data into two groups (Drinkers vs Non-Drinkers)
bmi_no_alc = df[df['Alcohol_Binary'] == 0]['BMIPCT'].dropna()
bmi_alc = df[df['Alcohol_Binary'] == 1]['BMIPCT'].dropna()

# 2. Perform Welch's T-test (assumes unequal variances for better accuracy)
t_stat_2, p_val_2 = stats.ttest_ind(bmi_no_alc, bmi_alc, equal_var=False)

# 3. Print Results
print(f"Mean BMI (Non-drinkers):  {bmi_no_alc.mean():.4f} (n={len(bmi_no_alc)})")
print(f"Mean BMI (Drinkers):      {bmi_alc.mean():.4f} (n={len(bmi_alc)})")
print(f"T-statistic:              {t_stat_2:.4f}")
print(f"P-value:                  {p_val_2:.4f}")

if p_val_2 < 0.05:
    print("\nDecision: Reject H0. There is a significant difference between the two groups.")
else:
    print("\nDecision: Fail to Reject H0. The observed difference is NOT statistically significant.")

=== Bonus Analysis: Two-Sample Comparison (Alcohol vs. No Alcohol) ===
Mean BMI (Non-drinkers):  64.7060 (n=6449)
Mean BMI (Drinkers):      64.7994 (n=5394)
T-statistic:              -0.1849
P-value:                  0.8533

Decision: Fail to Reject H0. The observed difference is NOT statistically significant.


### Interpretation: BMI Difference by Alcohol Use
* **Statistical Decision:** Fail to Reject the Null Hypothesis ($p = 0.8533$).
* **Practical Insight:** Although our EDA boxplot revealed that the alcohol-using group contains extreme right-skewed outliers (causing its mean to surpass its median), this two-sample test confirms that the overall mean difference between drinkers and non-drinkers is not statistically significant. The slight difference in averages is likely due to random variation.

In [5]:
# Export final summary results to a text file
summary_text = f"""
=== Final Project Cycle 2: Inference Summary ===

1. Proportion Analysis (Alcohol Use vs 0.35)
- P-value: {p_val_z:.4e}
- Decision: {'Reject H0' if p_val_z < 0.05 else 'Fail to Reject H0'}

2. Mean Analysis (BMI Percentile vs 65.0)
- P-value: {p_val_t:.4f}
- Decision: {'Reject H0' if p_val_t < 0.05 else 'Fail to Reject H0'}

3. Two-Sample T-test (BMI: Drinkers vs Non-Drinkers)
- P-value: {p_val_2:.4f}
- Decision: {'Reject H0' if p_val_2 < 0.05 else 'Fail to Reject H0'}
"""

with open('../outputs/inference/final_summary.txt', 'w') as f:
    f.write(summary_text)

print("✅ Successfully saved summary text to ../outputs/inference/final_summary.txt")

✅ Successfully saved summary text to ../outputs/inference/final_summary.txt


---
## Final Project Synthesis (Executive Summary)
Our statistical inference analysis reveals a critical disconnect between adolescent behavioral risks and physical health indicators:
1. **Behavioral Alert:** Alcohol consumption is significantly over-represented compared to expected norms, marking it as a primary area of concern.
2. **Physical Normality:** Conversely, the overall average BMI percentile remains statistically consistent with the 65.0 health benchmark.
3. **Nuanced Finding:** While alcohol use is prevalent, it does not statistically drive up the average BMI of the population, even though the drinking sub-group exhibits notable extreme weight outliers. 

**Recommendation:** Health interventions should prioritize behavioral corrections (alcohol use reduction) rather than broad-scale weight management, with targeted support for the extreme BMI outliers identified in the drinking group.